# 🏆 Exp C: Grandmaster Tabular ML Arena — Out-of-Fold Stacking (`XGBoost` + `LightGBM` + `CatBoost` + `PyTorch` Residual MLP), `SHAP` Explainability & Probability Calibration
**Author:** Akmal Khan, PhD  
**Repository:** [github.com/akmalkhaniub/enterprise-banking-ds-platform](https://github.com/akmalkhaniub/enterprise-banking-ds-platform)

### 🎯 Pipeline Highlights
1. **Weight of Evidence (WoE) & Information Value (IV)** Feature Screening (Basel III / IFRS-9 Risk Standard)
2. **5-Fold Stratified Out-of-Fold (OOF) Stacking** combining `LightGBM`, `XGBoost`, `CatBoost`, and a GPU `PyTorch` Residual Tabular Network
3. **Probability Calibration (`Isotonic` / Brier Score)** & **Population Stability Index (PSI) Drift Monitor**

In [ ]:
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss, average_precision_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# 1. Generate 12,000 Customer Credit & Churn Risk Profiles with Non-Linear Feature Interactions + Compute Information Value (IV)
n = 12000
debt_to_income = np.random.beta(2.2, 5.0, size=n)
utilization_ratio = np.random.beta(2.5, 3.2, size=n)
recency_days = np.random.exponential(scale=28.0, size=n)
txn_velocity_drop = np.random.normal(0.0, 0.35, size=n)
tenure_months = np.random.randint(3, 120, size=n).astype(float)
digital_engagement = np.random.uniform(0, 1, size=n)
late_payments_12m = np.random.poisson(lam=0.65, size=n).astype(float)
cash_advance_share = np.random.beta(1.5, 6.0, size=n)

logit = (
    -3.1
    + 2.8 * (debt_to_income ** 1.4)
    + 2.4 * (utilization_ratio * cash_advance_share * 2.5)
    + 0.022 * recency_days
    - 1.6 * txn_velocity_drop
    + 0.72 * late_payments_12m
    - 1.3 * digital_engagement
    - 0.012 * tenure_months
)
prob = 1.0 / (1.0 + np.exp(-logit))
y = (np.random.uniform(0, 1, size=n) < prob).astype(int)

df = pd.DataFrame({
    'debt_to_income': debt_to_income,
    'utilization_ratio': utilization_ratio,
    'recency_days': recency_days,
    'txn_velocity_drop': txn_velocity_drop,
    'tenure_months': tenure_months,
    'digital_engagement': digital_engagement,
    'late_payments_12m': late_payments_12m,
    'cash_advance_share': cash_advance_share,
    'util_x_cash_interaction': utilization_ratio * cash_advance_share,
    'dti_per_tenure': debt_to_income / (tenure_months + 1.0)
})

# Compute Information Value (IV) across decile bins
def calc_iv(series, target, bins=10):
    b = pd.qcut(series, q=bins, duplicates='drop')
    grouped = pd.DataFrame({'bin': b, 'y': target}).groupby('bin', observed=False)['y'].agg(['count', 'sum'])
    grouped['good'] = grouped['count'] - grouped['sum']
    dist_bad = (grouped['sum'] + 0.5) / (grouped['sum'].sum() + 0.5)
    dist_good = (grouped['good'] + 0.5) / (grouped['good'].sum() + 0.5)
    woe = np.log(dist_good / dist_bad)
    return float(((dist_good - dist_bad) * woe).sum())

iv_table = pd.DataFrame([{'Feature': c, 'Information_Value_IV': round(calc_iv(df[c], y), 4)} for c in df.columns]).sort_values('Information_Value_IV', ascending=False)
print('=== BASEL / CREDIT SCORECARD INFORMATION VALUE (IV) SCREENING ===')
print(iv_table.to_string(index=False))

In [ ]:
# 2. 5-Fold Out-of-Fold (OOF) Stacking: LightGBM + XGBoost + CatBoost + PyTorch Residual Tabular Net
X_train, X_test, y_train, y_test = train_test_split(df.values.astype(np.float32), y, test_size=0.25, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

class ResidualTabularMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.in_proj = nn.Linear(in_dim, 64)
        self.block = nn.Sequential(nn.Linear(64, 64), nn.BatchNorm1d(64), nn.SiLU(), nn.Dropout(0.15), nn.Linear(64, 64), nn.BatchNorm1d(64))
        self.head = nn.Linear(64, 1)
    def forward(self, x):
        h = torch.relu(self.in_proj(x))
        h = h + self.block(h)
        return self.head(h)

def fit_predict_pytorch_mlp(X_tr, y_tr, X_val, X_te):
    model = ResidualTabularMLP(X_tr.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    loader = DataLoader(TensorDataset(torch.tensor(X_tr).to(device), torch.tensor(y_tr, dtype=torch.float32).unsqueeze(-1).to(device)), batch_size=256, shuffle=True)
    model.train()
    for _ in range(18):
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        p_val = torch.sigmoid(model(torch.tensor(X_val).to(device))).squeeze(-1).cpu().numpy()
        p_te = torch.sigmoid(model(torch.tensor(X_te).to(device))).squeeze(-1).cpu().numpy()
    return p_val, p_te

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros((len(X_train), 4))
test_preds = np.zeros((len(X_test), 4))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_va, y_va = X_train[val_idx], y_train[val_idx]

    m_lgb = lgb.LGBMClassifier(n_estimators=220, learning_rate=0.04, num_leaves=31, random_state=42 + fold, verbose=-1)
    m_lgb.fit(X_tr, y_tr)
    oof_preds[val_idx, 0] = m_lgb.predict_proba(X_va)[:, 1]
    test_preds[:, 0] += m_lgb.predict_proba(X_test)[:, 1] / 5.0

    m_xgb = xgb.XGBClassifier(n_estimators=220, learning_rate=0.04, max_depth=4, eval_metric='logloss', random_state=42 + fold)
    m_xgb.fit(X_tr, y_tr)
    oof_preds[val_idx, 1] = m_xgb.predict_proba(X_va)[:, 1]
    test_preds[:, 1] += m_xgb.predict_proba(X_test)[:, 1] / 5.0

    m_cat = CatBoostClassifier(iterations=250, learning_rate=0.05, depth=5, verbose=0, random_seed=42 + fold)
    m_cat.fit(X_tr, y_tr)
    oof_preds[val_idx, 2] = m_cat.predict_proba(X_va)[:, 1]
    test_preds[:, 2] += m_cat.predict_proba(X_test)[:, 1] / 5.0

    p_va_mlp, p_te_mlp = fit_predict_pytorch_mlp(X_train_sc[tr_idx], y_tr, X_train_sc[val_idx], X_test_sc)
    oof_preds[val_idx, 3] = p_va_mlp
    test_preds[:, 3] += p_te_mlp / 5.0

# Meta-Learner Level-2 Stacking + Isotonic Probability Calibration
meta_lr = CalibratedClassifierCV(LogisticRegression(), method='isotonic', cv=3)
meta_lr.fit(oof_preds, y_train)
stacked_test_probs = meta_lr.predict_proba(test_preds)[:, 1]

names = ['1. LightGBM (5-Fold Bagged)', '2. XGBoost (5-Fold Bagged)', '3. CatBoost (5-Fold Bagged)', '4. PyTorch Residual Tabular MLP (GPU)']
rows = []
for idx, name in enumerate(names):
    rows.append({
        'Model': name,
        'OOF ROC-AUC': round(roc_auc_score(y_train, oof_preds[:, idx]), 4),
        'Holdout ROC-AUC': round(roc_auc_score(y_test, test_preds[:, idx]), 4),
        'Holdout PR-AUC': round(average_precision_score(y_test, test_preds[:, idx]), 4),
        'Brier Calibration Loss': round(brier_score_loss(y_test, test_preds[:, idx]), 4)
    })
rows.append({
    'Model': '5. Level-2 Calibrated Stacking Ensemble (All 4)',
    'OOF ROC-AUC': round(roc_auc_score(y_train, meta_lr.predict_proba(oof_preds)[:, 1]), 4),
    'Holdout ROC-AUC': round(roc_auc_score(y_test, stacked_test_probs), 4),
    'Holdout PR-AUC': round(average_precision_score(y_test, stacked_test_probs), 4),
    'Brier Calibration Loss': round(brier_score_loss(y_test, stacked_test_probs), 4)
})
print('=== 5-FOLD GRANDMASTER STACKING & CALIBRATION LEADERBOARD ===')
print(pd.DataFrame(rows).sort_values('Holdout ROC-AUC', ascending=False).to_string(index=False))